In [ ]:

!pip install pyswarms pygad

In [ ]:
import numpy as np 
import pandas as pd 
import os

# This lists the exact path to your data
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv('/kaggle/input/datasets/shreyasinha/dataset-containing-antenna-parameters/dataset_antenna.csv')
sns.heatmap(df.corr(), annot=True, cmap='coolwarm')
plt.title("Correlation between Antenna Parameters")
plt.show()

In [ ]:
print(df.columns.tolist())

In [ ]:
# 1. Define Inputs (Your design parameters)
# We exclude 'Freq(GHz)' if you want to optimize for a specific frequency, 
# or include it if the dataset covers multiple frequencies.
feature_cols = [
    'length of patch in mm', 
    'width of patch in mm', 
    'Slot length in mm', 
    'slot width in mm'
]

X = df[feature_cols].values

# 2. Define Target (What we want to minimize)
y = df['s11(dB)'].values

print("Features selected successfully!")

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.preprocessing import MinMaxScaler
from scipy.optimize import differential_evolution
import matplotlib.pyplot as plt

# 1. LOAD DATA
# Ensure your path is correct from the 'Add Data' sidebar
df = pd.read_csv('/kaggle/input/datasets/shreyasinha/dataset-containing-antenna-parameters/dataset_antenna.csv')

# 2. SELECT COLUMNS (Using your exact names)
feature_cols = [
    'length of patch in mm', 
    'width of patch in mm', 
    'Slot length in mm', 
    'slot width in mm'
]
X_raw = df[feature_cols].values
y = df['s11(dB)'].values

# 3. SCALE DATA (Fixes the NameError)
scaler_X = MinMaxScaler()
X_scaled = scaler_X.fit_transform(X_raw)

# 4. TRAIN THE SURROGATE MODEL
print("Training AI model... please wait.")
surrogate_model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6)
surrogate_model.fit(X_scaled, y)
print(f"Model Training Complete. Accuracy (R2): {surrogate_model.score(X_scaled, y):.4f}")

# 5. DEFINE OPTIMIZATION
def objective_function(scaled_params):
    # The optimizer tries different numbers; the AI predicts the S11
    prediction = surrogate_model.predict(scaled_params.reshape(1, -1))
    return prediction[0]

# Set bounds based on the min/max of your actual dataset
bounds = [
    (X_scaled[:,0].min(), X_scaled[:,0].max()), 
    (X_scaled[:,1].min(), X_scaled[:,1].max()), 
    (X_scaled[:,2].min(), X_scaled[:,2].max()), 
    (X_scaled[:,3].min(), X_scaled[:,3].max())
]

# 6. RUN OPTIMIZER
print("Searching for optimal dimensions...")
result = differential_evolution(objective_function, bounds, strategy='best1bin', maxiter=50)

# 7. CONVERT BACK TO MM
optimized_mm = scaler_X.inverse_transform(result.x.reshape(1, -1))

# --- FINAL RESULTS ---
print("\n" + "="*30)
print("   OPTIMIZED DESIGN FOUND")
print("="*30)
print(f"Length of Patch: {optimized_mm[0][0]:.3f} mm")
print(f"Width of Patch:  {optimized_mm[0][1]:.3f} mm")
print(f"Slot Length:     {optimized_mm[0][2]:.3f} mm")
print(f"Slot Width:      {optimized_mm[0][3]:.3f} mm")
print(f"Predicted S11:   {result.fun:.2f} dB")
print("="*30)

In [ ]:
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

# 1. SPLIT THE DATA (This fixes the NameError)
# We take 80% for training and 20% for testing
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# 2. RE-TRAIN ON THE SPLIT (To ensure we can validate)
surrogate_model.fit(X_train, y_train)
y_pred = surrogate_model.predict(X_test)

# --- PLOT 1: ACCURACY (Actual vs Predicted) ---
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.scatter(y_test, y_pred, alpha=0.6, color='#1f77b4', edgecolors='k')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual S11 (dB)')
plt.ylabel('Predicted S11 (dB)')
plt.title('AI Prediction Accuracy')
plt.grid(True, linestyle='--', alpha=0.7)

# --- PLOT 2: ERROR DISTRIBUTION ---
plt.subplot(1, 2, 2)
errors = y_test - y_pred
sns.histplot(errors, kde=True, color='green')
plt.title('Prediction Error Distribution')
plt.xlabel('Error (dB)')

plt.tight_layout()
plt.show()

# --- PLOT 3: PARAMETER SENSITIVITY ---
plt.figure(figsize=(8, 4))
importances = surrogate_model.feature_importances_
indices = np.argsort(importances)

plt.barh(range(len(indices)), importances[indices], color='darkorange', align='center')
plt.yticks(range(len(indices)), [feature_cols[i] for i in indices])
plt.xlabel('Importance Score (Gain)')
plt.title('Which Parameter Affects S11 the Most?')
plt.show()

In [ ]:
# Create a dictionary of the results
optimized_data = {
    'Parameter': feature_cols,
    'Optimized Value (mm)': optimized_mm[0],
    'Predicted S11 (dB)': [result.fun] * len(feature_cols)
}

# Convert to a DataFrame
results_df = pd.DataFrame(optimized_data)

# Save to the Kaggle working directory
results_df.to_csv('optimized_antenna_design.csv', index=False)

print(" Success! Your optimized design has been saved.")
print("Look at the 'Output' section in the right-hand sidebar of Kaggle to download 'optimized_antenna_design.csv'.")

In [ ]:
# Select the parameter you want to sweep (e.g., 'Slot length in mm')
sweep_feature_index = 2 
feature_name = feature_cols[sweep_feature_index]

# Create a range of values around the optimized point (+/- 20%)
optimized_point = result.x.copy()
sweep_range = np.linspace(bounds[sweep_feature_index][0], bounds[sweep_feature_index][1], 100)

sweep_predictions = []
for val in sweep_range:
    temp_point = optimized_point.copy()
    temp_point[sweep_feature_index] = val
    sweep_predictions.append(objective_function(temp_point))

# Plotting the Sweep
plt.figure(figsize=(8, 5))
# Scale back to mm for the x-axis
real_mm_range = np.linspace(X_raw[:, sweep_feature_index].min(), X_raw[:, sweep_feature_index].max(), 100)

plt.plot(real_mm_range, sweep_predictions, color='purple', lw=2)
plt.axvline(optimized_mm[0][sweep_feature_index], color='red', linestyle='--', label='Optimized Point')
plt.xlabel(f'{feature_name}')
plt.ylabel('Predicted S11 (dB)')
plt.title(f'Sensitivity Analysis: Effect of {feature_name} on Resonance')
plt.legend()
plt.grid(True)
plt.show()